# 365 Probabilidades - Dia #105
## Qual a probabilidade de uma música te dar arrepio?

**Tipo:** Descritivo
**Data de publicação:** 2026-09-26
**Ferramenta:** Python
**Decisão analisada:** O arrepio que a música me dá é comum ou é coisa minha?
**Hashtag:** #365Probabilidades #Dia105

---

### 📖 A História

Recentemente, no dia **#084**, falei sobre gosto musical. Hoje quero falar sobre a sensação que a música traz.

Aquele segundo em que a música vira, a voz entra, um instrumento aparece e alguma coisa sobe pela espinha. Dura três, quatro segundos. Some. E, às vezes, deixa a pessoa parada no meio do que estava fazendo. Ou, no meu caso, com o pensamento absorto por alguns segundos.

A gente costuma tratar isso como um detalhe sem importância ou como sinal de sensibilidade demais. Quando acontece comigo, por exemplo, sempre penso que devo estar em um daqueles meus momentos mais sensíveis.

E aí, fui ver se isso é algo que também acontece com outras pessoas. E com que frequência.

---

### 📚 O Conceito

O fenômeno tem nome na literatura: arrepio evocado por música. Ele aparece com uma dúzia de nomes diferentes, de arrepio na espinha a pele arrepiada, e não é tratado como detalhe pelos pesquisadores: o arrepio é usado como marcador de prazer musical porque é discreto, memorável e, quando vem com pele arrepiada, observável de fora.

A pergunta interessante não é só quanta gente sente. É **onde** o arrepio acontece. Se a sensação fosse sobre sensibilidade pessoal, ela apareceria espalhada pela música. Não é o que se observa: os arrepios se concentram em momentos identificáveis, e dá para provocá-los, ou apagá-los, mexendo na gravação.

Este dia é a continuação do **#084**, que perguntou se o gosto musical foi decidido aos 14 anos. Lá a pergunta era por que certas músicas grudam. Aqui é o que elas fazem com o corpo.

---

### 🧮 O Modelo

Três camadas.

**1. Quem sente.** Cinco pesquisas populacionais, reunidas num modelo hierárquico: cada uma estima a própria prevalência, e todas compartilham uma média comum. Assim a estimativa final respeita tanto o tamanho de cada amostra quanto a diferença entre elas, que é real. Somar tudo num único percentual esconderia justamente essa diferença.

**2. Com que frequência.** Um estudo de amostragem de experiência dá a probabilidade por ocasião de escuta. Com ela, uma simulação responde à pergunta prática: quantas vezes até o próximo arrepio.

**3. Não é você, é o segundo.** Dois experimentos de laboratório mexem na própria gravação: um remove o trecho que causa o arrepio, o outro sobe e desce o volume em 6 dB no ponto exato do clímax. Uma regressão de Poisson mede quanto o arrepio responde a cada decibel, e uma peça de controle, sem clímax estrutural, serve de contraprova.

**Fontes:**
- de Fleurian, R. & Pearce, M. T. (2021). "Chills in Music: A Systematic Review". *Psychological Bulletin*, 147(9), 890-920. Revisão sistemática de **167 trabalhos** publicados entre 1980 e 2020. É a fonte guarda-chuva: reúne as prevalências, cataloga os gatilhos e explica por que não fez meta-análise.
- Goldstein, A. (1980), **N=249**; Sloboda, J. (1991), **N=83**; Panksepp, J. (1995), **N=828**; Nusbaum, E. C. & Silvia, P. J. (2011), **N=196**; Mlejnek, R. (2013), **N=186**. Surveys retrospectivos de prevalência, reunidos pela revisão acima. Somados, **N=1.542**.
- Nusbaum, E. C., Silvia, P. J., Beaty, R. E., Burgin, C. J., Hodges, D. A. & Kwapil, T. R. (2014). "Listening between the notes: Aesthetic chills in everyday music listening". *Psychology of Aesthetics, Creativity, and the Arts*, 8(1), 104-109. **Amostragem de experiência: 106 universitários, 10 avisos por dia durante uma semana.**
- Bannister, S. & Eerola, T. (2018). "Suppressing the Chills: Effects of Musical Manipulation on the Chills Response". *Frontiers in Psychology*, 9:2046. **N=24.** Três peças, cada uma em versão original e em versão com o trecho de arrepio removido.
- Bannister, S. (2020). "A Vigilance Explanation of Musical Chills? Effects of Loudness and Brightness Manipulations". *Music & Science*, 3. **N=40, 34 com dados de condutância aproveitáveis.** Volume e brilho manipulados em ±6 dBA numa janela de 8 segundos no pico estrutural.
- Contexto genético, só neste notebook: Bignardi, G., Chamberlain, R., Kevenaar, S. T., Tamimy, Z. & Boomsma, D. I. (2022), *Scientific Reports* 12:3247, desenho clássico de gêmeos com mais de 10.000 holandeses; e Bignardi, G., Admiraal, D., Eising, E. & Fisher, S. E. (2026), *PLOS Genetics* 22(2):e1012002, **n=15.606** genotipados da coorte Lifelines.

**Fator ×0,80:** aplicado a **todas as proporções de survey autorrelatado** deste dia, que são as camadas 1 e 2: prevalência e frequência por ocasião. **Não** aplicado à camada 3, e o motivo é a própria regra do projeto: ali não há proporção de survey, e sim contagem de episódios de arrepio registrados por botão em laboratório, com condutância da pele validando o relato. São medidas objetivas de experimento controlado, categoria explicitamente excluída do fator.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize
from scipy.special import expit, logit

SEED = 42
rng = np.random.default_rng(SEED)

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

DOURADO, VERMELHO, VERDE, CINZA = '#c9a14a', '#c0392b', '#2a8a82', '#6b6a64'

print("✅ Bibliotecas carregadas · semente", SEED)


In [ ]:
# --- DADOS DA LITERATURA ---
# Guarda-chuva: de Fleurian & Pearce (2021), "Chills in Music: A Systematic
# Review", Psychological Bulletin 147(9), 890-920. Revisão sistemática de 167
# trabalhos (1980-2020). Não é meta-análise: os próprios autores dizem que a
# diversidade de métodos inviabiliza agregação quantitativa.

FATOR_CORRECAO = 0.80   # só nas camadas 1 e 2 (proporções de survey autorrelatado)

# ── CAMADA 1 · prevalência ───────────────────────────────────────────────
# nome, ano, N, proporção relatada, o que exatamente foi perguntado
PESQUISAS = [
    ("Goldstein",        1980, 249, 0.79, "já sentiu arrepio com música"),
    ("Sloboda",          1991,  83, 0.90, "arrepio na espinha, ao menos 1x em 5 anos"),
    ("Panksepp",         1995, 828, 0.86, "sente com alguma regularidade"),
    ("Nusbaum & Silvia", 2011, 196, 0.92, "100% menos os 8% que nunca ou raramente"),
    ("Mlejnek",          2013, 186, 0.80, "espinha ou pele, ao menos raramente, em 5 anos"),
]
NOMES = [f"{a} {b}" for a, b, *_ in PESQUISAS]
N_EST = np.array([p[2] for p in PESQUISAS])
P_EST = np.array([p[3] for p in PESQUISAS])
X_EST = np.round(N_EST * P_EST).astype(int)

# ── CAMADA 2 · frequência no dia a dia ───────────────────────────────────
# Nusbaum, Silvia, Beaty, Burgin, Hodges & Kwapil (2014), Psychology of
# Aesthetics, Creativity, and the Arts 8(1), 104-109: amostragem de experiência,
# 106 universitários, 10 avisos por dia durante uma semana.
N_ESP = 106
P_OCASIAO = 0.14        # arrepio em 14% das ocasiões em que ouviam música
P_SEMANA = 0.81         # 81% tiveram ao menos um episódio na semana

# ── CAMADA 3 · onde o arrepio acontece ───────────────────────────────────
# (a) O catálogo de gatilhos, de Fleurian & Pearce (2021), Tabela 5:
#     número de estudos, dos 167 revisados, que apontam cada gatilho.
GATILHOS = [
    ("Mudança de textura, entrada de instrumento", 8),
    ("Crescendo, build-up, clímax",                7),
    ("Mudança de volume",                          7),
    ("Mudança de melodia ou harmonia",             6),
    ("Dissonância, aspereza",                      5),
    ("Picos de volume",                            5),
    ("Brilho, agudos",                             4),
    ("Voz e letra",                                2),
]

# (b) Bannister & Eerola (2018), Frontiers in Psychology 9:2046, N=24.
#     Três peças, cada uma em duas versões: original e com o trecho de
#     arrepio removido. Contagem de episódios de arrepio relatados.
REMOCAO = [
    ("Glósóli · Sigur Rós",        10,  7, "trecho de 4:34 em diante"),
    ("Jupiter · Gustav Holst",     10,  8, "trecho de 3:09 a 4:55"),
    ("Ancestral · Steven Wilson",  10,  6, "trecho de 4:02 a 5:02"),
]
N_REMOCAO = 24
# O teste dos autores, que respeita o desenho pareado: chi² = 3,85 · p = 0,049
CHI2_AUTORES, P_AUTORES = 3.85, 0.049

# (c) Bannister (2020), Music & Science 3, N=40 (34 com condutância aproveitável).
#     Volume manipulado em ±6 dBA numa janela de 8 s no pico estrutural.
#     Contagem de episódios de arrepio por condição.
DB = np.array([-6.0, 0.0, 6.0])
CHILLS_PICO     = np.array([ 6.0, 11.0, 29.0])   # Glósóli, tem clímax estrutural
CHILLS_CONTROLE = np.array([11.0,  5.0, 10.0])   # Ancestral, solo de guitarra
NOME_PICO, NOME_CONTROLE = "Glósóli (com clímax)", "Ancestral (sem clímax)"
N_BANNISTER = 34

# ── Genética (só contexto, não entra no Substack) ────────────────────────
# Bignardi, Chamberlain, Kevenaar, Tamimy & Boomsma (2022), Scientific Reports
#   12:3247 — desenho clássico de gêmeos, mais de 10.000 holandeses: h² de 36% a 43%
# Bignardi, Admiraal, Eising & Fisher (2026), PLOS Genetics 22(2):e1012002 —
#   n=15.606 genotipados (coorte Lifelines): até 29% da variação explicada por
#   parentesco, um quarto disso atribuível a variantes comuns de DNA;
#   correlação genética de 0,55 entre arrepio de arte e arrepio de música
H2_GEMEOS = (0.36, 0.43)
H2_PEDIGREE_2026 = 0.29
COR_GENETICA = 0.55

mil = lambda n: f"{int(n):,}".replace(',', '.')

print("=" * 70)
print("  CAMADA 1 · QUEM SENTE ARREPIO COM MÚSICA")
print("=" * 70)
print(f"  {'Pesquisa':<20} {'N':>6} {'Relatado':>9}   Pergunta")
for (nome, ano, n, p, q), x in zip(PESQUISAS, X_EST):
    print(f"  {nome + ' ' + str(ano):<20} {mil(n):>6} {p:>8.0%}   {q}")
print(f"\n  Somando as cinco amostras: N = {mil(N_EST.sum())}")
print(f"  Dispersão entre elas: de {P_EST.min():.0%} a {P_EST.max():.0%}")

print("\n" + "=" * 70)
print(f"  CAMADA 2 · FREQUÊNCIA (Nusbaum et al. 2014, N={N_ESP}, 1 semana)")
print("=" * 70)
print(f"  → arrepio em {P_OCASIAO:.0%} das ocasiões de escuta")
print(f"  → {P_SEMANA:.0%} tiveram ao menos um episódio na semana")

print("\n" + "=" * 70)
print("  CAMADA 3 · ONDE O ARREPIO ACONTECE")
print("=" * 70)
print("  Gatilhos catalogados pela revisão (nº de estudos, dos 167):")
for nome, k in GATILHOS:
    print(f"    {k:>2}  {'█' * k}  {nome}")
print(f"\n  Experimento de remoção (Bannister & Eerola 2018, N={N_REMOCAO}):")
print(f"    {'Peça':<30} {'original':>9} {'editada':>9}   {'o que saiu'}")
for nome, o, e, trecho in REMOCAO:
    print(f"    {nome:<30} {o:>9.0f} {e:>9.0f}   {trecho}")
tot_o = sum(r[1] for r in REMOCAO); tot_e = sum(r[2] for r in REMOCAO)
print(f"    {'TOTAL':<30} {tot_o:>9.0f} {tot_e:>9.0f}")
print(f"\n  Experimento de volume (Bannister 2020, N={N_BANNISTER}, ±6 dBA no pico):")
print(f"    {'Condição':<24} {'-6 dB':>7} {'original':>9} {'+6 dB':>7}")
print(f"    {NOME_PICO:<24} {CHILLS_PICO[0]:>7.0f} {CHILLS_PICO[1]:>9.0f} {CHILLS_PICO[2]:>7.0f}")
print(f"    {NOME_CONTROLE:<24} {CHILLS_CONTROLE[0]:>7.0f} {CHILLS_CONTROLE[1]:>9.0f} {CHILLS_CONTROLE[2]:>7.0f}")
print(f"\n  ×0,80 aplicado nas camadas 1 e 2 (survey autorrelatado).")
print(f"  Camada 3 fora do fator: contagem de laboratório com condutância da pele.")
print("=" * 70)


In [ ]:
# --- O MODELO ---
# Camada 1: modelo hierárquico das cinco pesquisas
# Camada 2: Monte Carlo do próximo arrepio
# Camada 3: regressão de Poisson do arrepio contra decibéis

# ---------------------------------------------------------------------------
# CAMADA 1 — hierárquico
# theta_i = prevalência verdadeira de cada pesquisa
# logit(theta_i) ~ Normal(mu, tau²)   ·   x_i ~ Binomial(n_i, theta_i)
# Priores: mu ~ Normal(0, 2²) na escala logit, tau ~ meia-normal(0,5)
# Posterior por grade, com a integral em theta feita por Gauss-Hermite.

nos, pesos = np.polynomial.hermite_e.hermegauss(80)   # para Normal(0,1)
mu_grid = np.linspace(logit(0.50), logit(0.985), 240)
tau_grid = np.linspace(0.01, 2.0, 200)
MU, TAU = np.meshgrid(mu_grid, tau_grid, indexing='ij')

log_post = (stats.norm.logpdf(MU, 0, 2)
            + stats.halfnorm.logpdf(TAU, scale=0.5))
for n_i, x_i in zip(N_EST, X_EST):
    theta = expit(MU[..., None] + TAU[..., None] * nos)          # (mu, tau, nó)
    vero = stats.binom.pmf(x_i, n_i, theta) @ (pesos / np.sqrt(2 * np.pi))
    log_post += np.log(vero)

post = np.exp(log_post - log_post.max())
post /= post.sum()

idx = rng.choice(post.size, size=200_000, p=post.ravel())
mu_s = MU.ravel()[idx]
tau_s = TAU.ravel()[idx]

prev_tipica = expit(mu_s)                                   # prevalência típica
theta_novo = expit(mu_s + tau_s * rng.standard_normal(mu_s.size))  # nova população
prev_corrigida = prev_tipica * FATOR_CORRECAO

q = lambda a: np.percentile(a, [2.5, 50, 97.5])
q_tip, q_novo, q_corr = q(prev_tipica), q(theta_novo), q(prev_corrigida)
pool_ingenuo = X_EST.sum() / N_EST.sum()

print("=" * 70)
print("  CAMADA 1 · QUANTA GENTE SENTE, JUNTANDO AS CINCO PESQUISAS")
print("=" * 70)
print(f"  Pool ingênuo (somar tudo):      {pool_ingenuo:.1%}   ← ignora a heterogeneidade")
print(f"  Hierárquico, prevalência típica: {q_tip[1]:.1%}  IC 95% [{q_tip[0]:.1%}, {q_tip[2]:.1%}]")
print(f"  Previsão para uma nova amostra:  {q_novo[1]:.1%}  IC 95% [{q_novo[0]:.1%}, {q_novo[2]:.1%}]")
print(f"  Heterogeneidade tau (logit):     {np.median(tau_s):.2f}  IC 95% [{np.percentile(tau_s,2.5):.2f}, {np.percentile(tau_s,97.5):.2f}]")
print("\n  Com ×0,80 (autorrelato):")
print(f"  → {q_corr[1]:.1%}  IC 95% [{q_corr[0]:.1%}, {q_corr[2]:.1%}]")
print(f"  → ou seja, cerca de {q_corr[1]*10:.0f} em cada 10 pessoas")

# ---------------------------------------------------------------------------
# CAMADA 2 — Monte Carlo do próximo arrepio
# Cada ocasião de escuta é um ensaio com probabilidade p. Quantas ocasiões
# até o próximo arrepio? Distribuição geométrica, simulada.
p_bruto = P_OCASIAO
p_corr = P_OCASIAO * FATOR_CORRECAO
R = 200_000

print("\n" + "=" * 70)
print("  CAMADA 2 · QUANTAS VEZES ATÉ O PRÓXIMO ARREPIO")
print("=" * 70)
espera = {}
for nome, p in [("sem correção (14%)", p_bruto), ("com ×0,80 (11,2%)", p_corr)]:
    sim = rng.geometric(p, R)
    espera[nome] = (np.median(sim), np.percentile(sim, 90), sim.mean())
    print(f"  {nome:<20} mediana {np.median(sim):.0f} ocasiões · "
          f"90% em até {np.percentile(sim, 90):.0f} · média {sim.mean():.1f}")
print(f"\n  Traduzindo: uma em cada {1/p_corr:.0f} vezes que você põe música para tocar")
for n_oc in (5, 10, 20, 50):
    print(f"  P(ao menos um arrepio em {n_oc:>2} ocasiões) = {1-(1-p_corr)**n_oc:.0%}")
k_equiv = np.log(1 - P_SEMANA) / np.log(1 - p_corr)
print(f"\n  Conferência de coerência com a semana observada:")
print(f"  → observado: {P_SEMANA:.0%} tiveram ao menos um episódio em uma semana")
print(f"  → com {p_corr:.1%} por ocasião, isso equivale a ~{k_equiv:.0f} ocasiões de escuta na semana")
print(f"     (cerca de duas por dia), o que é plausível para quem ouve música no dia a dia")

# ---------------------------------------------------------------------------
# CAMADA 3 — não é você, é o segundo
#
# (a) Remoção do trecho. Dos episódios de arrepio somados nas duas versões,
#     quantos caíram na versão editada? Sem efeito, a expectativa é metade.
#     Teste binomial exato e IC de Jeffreys na fração.
tot_orig = sum(r[1] for r in REMOCAO)
tot_edit = sum(r[2] for r in REMOCAO)
tot = tot_orig + tot_edit
bt = stats.binomtest(int(tot_edit), int(tot), 0.5)
frac = tot_edit / tot
ic_frac = stats.beta(tot_edit + 0.5, tot_orig + 0.5).interval(0.95)
queda = 1 - tot_edit / tot_orig

print("\n" + "=" * 70)
print("  CAMADA 3a · TIRAR O TRECHO TIRA O ARREPIO?")
print("=" * 70)
print(f"  Episódios na versão original: {tot_orig:.0f}")
print(f"  Episódios na versão editada:  {tot_edit:.0f}   → queda de {queda:.0%}")
print(f"  Fração na versão editada: {frac:.1%}  IC 95% [{ic_frac[0]:.1%}, {ic_frac[1]:.1%}]")
print(f"  Teste binomial exato contra 50%: p = {bt.pvalue:.3f}")
print(f"\n  ATENÇÃO À DIFERENÇA DE TESTE:")
print(f"  Os autores, com teste que respeita o desenho pareado (cada pessoa ouve")
print(f"  as duas versões), relatam chi² = {CHI2_AUTORES} · p = {P_AUTORES}.")
print(f"  O teste simples acima, que trata os episódios como independentes,")
print(f"  dá p = {bt.pvalue:.2f} e não é significativo. A direção é a mesma nas")
print(f"  três peças, mas com N={N_REMOCAO} a evidência é frágil. Fica declarado.")

# (b) Dose-resposta do volume. Regressão de Poisson log-linear:
#     log(E[contagem]) = a + b · dB.   exp(b) = multiplicador por decibel.
#     Ajuste por máxima verossimilhança, EP pela informação observada.
def poisson_loglinear(x, y):
    X = np.column_stack([np.ones_like(x), x])
    nll = lambda b: np.sum(np.exp(X @ b) - y * (X @ b))
    res = optimize.minimize(nll, [np.log(y.mean()), 0.0], method='BFGS')
    b = res.x
    mu = np.exp(X @ b)
    cov = np.linalg.inv(X.T @ (X * mu[:, None]))
    return b, np.sqrt(np.diag(cov)), mu

print("\n" + "=" * 70)
print("  CAMADA 3b · QUANTO O ARREPIO RESPONDE A CADA DECIBEL")
print("=" * 70)
POISSON = {}
for nome, y in [(NOME_PICO, CHILLS_PICO), (NOME_CONTROLE, CHILLS_CONTROLE)]:
    b, se, mu = poisson_loglinear(DB, y)
    z = b[1] / se[1]
    pval = 2 * stats.norm.sf(abs(z))
    m6 = np.exp(b[1] * 6)
    lo6, hi6 = np.exp((b[1] - 1.96 * se[1]) * 6), np.exp((b[1] + 1.96 * se[1]) * 6)
    POISSON[nome] = dict(b=b, se=se, mu=mu, mult_db=np.exp(b[1]),
                         m6=m6, ic6=(lo6, hi6), p=pval, z=z)
    print(f"\n  {nome}")
    print(f"    observado:  -6 dB = {y[0]:.0f}  ·  original = {y[1]:.0f}  ·  +6 dB = {y[2]:.0f}")
    print(f"    ajustado:   {mu[0]:.1f}  ·  {mu[1]:.1f}  ·  {mu[2]:.1f}")
    print(f"    multiplicador por decibel: {np.exp(b[1]):.3f}×")
    print(f"    efeito de +6 dB:           {m6:.2f}×   IC 95% [{lo6:.2f}×, {hi6:.2f}×]")
    print(f"    z = {z:+.2f} · p = {pval:.4f}")

pico, ctrl = POISSON[NOME_PICO], POISSON[NOME_CONTROLE]
print(f"\n  A CONTRAPROVA:")
print(f"  Na peça com clímax estrutural, subir 6 dB multiplica o arrepio por {pico['m6']:.2f}.")
print(f"  Na peça sem clímax, o mesmo aumento dá {ctrl['m6']:.2f}× (IC [{ctrl['ic6'][0]:.2f}, {ctrl['ic6'][1]:.2f}]),")
print(f"  ou seja, nada. Não é volume: é volume na hora certa.")

print("\n" + "=" * 70)
print("  CONTEXTO GENÉTICO (fica no notebook, fora do Substack)")
print("=" * 70)
print(f"  Bignardi et al. 2022 (Sci Rep, gêmeos, >10.000 holandeses):")
print(f"  → h² entre {H2_GEMEOS[0]:.0%} e {H2_GEMEOS[1]:.0%}, sem efeito de ambiente compartilhado")
print(f"  Bignardi et al. 2026 (PLOS Genetics, n=15.606 genotipados):")
print(f"  → até {H2_PEDIGREE_2026:.0%} da variação explicada por parentesco,")
print(f"     um quarto disso atribuível a variantes comuns de DNA")
print(f"  → correlação genética de {COR_GENETICA:.2f} entre arrepio de arte e de música")

print("\n" + "=" * 70)
print("  RESUMO PARA O INSIGHT")
print("=" * 70)
print(f"  Sentem arrepio: {q_tip[1]:.0%} típico · {q_corr[1]:.0%} com ×0,80 [{q_corr[0]:.0%}, {q_corr[2]:.0%}]")
print(f"  Nova amostra pode cair entre {q_novo[0]:.0%} e {q_novo[2]:.0%}")
print(f"  Frequência: 1 em cada {1/p_corr:.0f} ocasiões · mediana {espera['com ×0,80 (11,2%)'][0]:.0f} · 90% em até {espera['com ×0,80 (11,2%)'][1]:.0f}")
print(f"  Em 20 ocasiões, chance de ao menos um: {1-(1-p_corr)**20:.0%} · coerência com a semana: ~{k_equiv:.0f} ocasiões")
print(f"  Remoção do trecho: {tot_orig:.0f} → {tot_edit:.0f} episódios, queda de {queda:.0%} (p dos autores = {P_AUTORES})")
print(f"  ASSINATURA: +6 dB no clímax = {pico['m6']:.2f}× mais arrepio, IC 95% [{pico['ic6'][0]:.2f}, {pico['ic6'][1]:.2f}], p = {pico['p']:.4f}")
print(f"  Contraprova sem clímax: {ctrl['m6']:.2f}× (IC [{ctrl['ic6'][0]:.2f}, {ctrl['ic6'][1]:.2f}]), p = {ctrl['p']:.2f}")


In [ ]:
# --- VISUALIZAÇÃO ---
rodape = lambda t: plt.figtext(0.5, 0.005, t + ' | #365Probabilidades', ha='center', fontsize=9, color='gray')

# ── GRÁFICO 1 · as cinco pesquisas e a média hierárquica ──
fig1, ax1 = plt.subplots(figsize=(11, 6.5))
ys = np.arange(len(PESQUISAS))[::-1]
for y, (nome, ano, n, p, _) in zip(ys, PESQUISAS):
    x = round(p * n)
    lo, hi = stats.beta(x + 0.5, n - x + 0.5).interval(0.95)
    ax1.plot([lo * 100, hi * 100], [y, y], color=CINZA, lw=2)
    ax1.plot(p * 100, y, 'o', color=VERDE, ms=8 + 6 * n / N_EST.max())
    ax1.text(hi * 100 + 1.2, y, f"N = {mil(n)}", va='center', fontsize=10, color=CINZA)
ax1.axvspan(q_tip[0] * 100, q_tip[2] * 100, color=DOURADO, alpha=0.18)
ax1.axvline(q_tip[1] * 100, color=DOURADO, lw=2.5)
ax1.axvline(q_corr[1] * 100, color=VERMELHO, lw=2.5, ls='--')
ax1.text(q_tip[1] * 100, len(ys) - 0.4, f"  modelo: {q_tip[1]:.0%}", color=DOURADO, fontsize=11, fontweight='bold')
ax1.text(q_corr[1] * 100, len(ys) - 0.9, f"com ×0,80: {q_corr[1]:.0%}  ", color=VERMELHO,
         fontsize=11, fontweight='bold', ha='right')
ax1.set_yticks(ys, NOMES)
ax1.set_xlim(55, 102)
ax1.set_ylim(-0.7, len(ys) - 0.2)
ax1.set_xlabel('Pessoas que sentem arrepio com música (%)')
ax1.set_title('Cinco pesquisas, uma estimativa\nModelo hierárquico sobre '
              f'{mil(N_EST.sum())} respondentes · barras: IC 95% de cada pesquisa', fontsize=13, pad=12)
rodape('Fontes: revisão de de Fleurian & Pearce 2021 (Psychological Bulletin)')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-105-grafico-01-prevalencia.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ── GRÁFICO 2 · quantas ocasiões até o próximo arrepio ──
fig2, (a, b) = plt.subplots(1, 2, figsize=(12, 6))
sim = rng.geometric(p_corr, 200_000)
a.hist(sim[sim <= 40], bins=np.arange(1, 42) - 0.5, color=VERDE, alpha=0.6, edgecolor='white')
a.axvline(np.median(sim), color=VERMELHO, lw=2.5)
a.text(np.median(sim) + 0.6, a.get_ylim()[1] * 0.9, f'mediana: {np.median(sim):.0f}',
       color=VERMELHO, fontsize=11, fontweight='bold')
a.set_xlabel('Ocasiões de escuta até o próximo arrepio')
a.set_ylabel('Simulações (de 200.000)')
a.set_title('O próximo arrepio', fontsize=12)

n_oc = np.arange(0, 41)
b.plot(n_oc, (1 - (1 - p_corr) ** n_oc) * 100, color=DOURADO, lw=3, label='com ×0,80 (11,2%)')
b.plot(n_oc, (1 - (1 - p_bruto) ** n_oc) * 100, color=CINZA, lw=1.5, ls=':', label='sem correção (14%)')
for marca in (10, 20):
    b.plot(marca, (1 - (1 - p_corr) ** marca) * 100, 'o', color=VERMELHO, ms=8)
    b.text(marca + 0.8, (1 - (1 - p_corr) ** marca) * 100 - 5,
           f'{marca} ocasiões: {1-(1-p_corr)**marca:.0%}', fontsize=10, color=VERMELHO)
b.set_xlabel('Ocasiões de escuta')
b.set_ylabel('Chance de ao menos um arrepio (%)')
b.set_title('Acumulando as chances', fontsize=12)
b.legend(frameon=False, loc='lower right')
fig2.suptitle('Uma em cada nove vezes que você põe música para tocar\n'
              'Nusbaum et al. 2014 · 106 pessoas, 10 avisos por dia, uma semana', fontsize=13)
rodape('Fonte: Nusbaum et al. 2014, Psychology of Aesthetics, Creativity, and the Arts')
plt.tight_layout(rect=(0, 0.03, 1, 0.92))
plt.savefig('dia-105-grafico-02-proximo-arrepio.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ── GRÁFICO 3 · não é você, é o segundo ──
br = lambda v, d=2: f"{v:.{d}f}".replace('.', ',')
fig3, (c, d) = plt.subplots(1, 2, figsize=(13, 6.5))

# painel esquerdo: remoção do trecho
lbl = [r[0].split(' · ')[0] for r in REMOCAO]
orig = np.array([r[1] for r in REMOCAO], float)
edit = np.array([r[2] for r in REMOCAO], float)
xs = np.arange(len(REMOCAO))
c.bar(xs - 0.19, orig, width=0.36, color=VERDE, alpha=0.85, label='versão original')
c.bar(xs + 0.19, edit, width=0.36, color=VERMELHO, alpha=0.85, label='trecho do arrepio removido')
for x, o, e in zip(xs, orig, edit):
    c.text(x - 0.19, o + 0.35, f'{o:.0f}', ha='center', fontsize=12, fontweight='bold', color=VERDE)
    c.text(x + 0.19, e + 0.35, f'{e:.0f}', ha='center', fontsize=12, fontweight='bold', color=VERMELHO)
c.set_xticks(xs, lbl, fontsize=10)
c.set_ylim(0, 15.5)
c.set_ylabel('Episódios de arrepio relatados')
c.set_title(f'Tirar o trecho tira o arrepio\n{tot_orig:.0f} → {tot_edit:.0f} episódios, '
            f'queda de {queda:.0%} · N={N_REMOCAO}', fontsize=12, pad=10)
c.legend(frameon=False, fontsize=9.5, loc='upper center', ncol=2)

# painel direito: dose-resposta do volume
grade = np.linspace(-7.5, 7.5, 200)
for nome, y, cor, mk in [(NOME_PICO, CHILLS_PICO, DOURADO, 'o'),
                         (NOME_CONTROLE, CHILLS_CONTROLE, CINZA, 's')]:
    bb = POISSON[nome]['b']
    d.plot(grade, np.exp(bb[0] + bb[1] * grade), color=cor, lw=2.5,
           ls='-' if cor == DOURADO else '--')
    d.plot(DB, y, mk, color=cor, ms=11, mec='white', mew=1.5, label=nome, zorder=5)
for x, y in zip(DB, CHILLS_PICO):
    d.text(x, y + 1.6, f'{y:.0f}', ha='center', fontsize=12, fontweight='bold', color=DOURADO)
d.set_xticks(DB, ['-6 dB', 'original', '+6 dB'])
d.set_ylim(0, 35)
d.set_xlim(-8, 8)
d.set_ylabel('Episódios de arrepio relatados')
d.set_title('Volume no ponto do clímax\n'
            f"+6 dB = {br(pico['m6'])}× mais arrepio  ·  IC 95% "
            f"[{br(pico['ic6'][0])}×, {br(pico['ic6'][1])}×]", fontsize=12, pad=10)
d.legend(frameon=False, fontsize=9, loc='upper left')
d.text(0.97, 0.06, f"regressão de Poisson · {br(pico['mult_db'], 3)}× por decibel\n"
                   f"peça sem clímax: {br(ctrl['m6'])}× (p = {br(ctrl['p'])})",
       transform=d.transAxes, ha='right', fontsize=9, color=CINZA)

fig3.suptitle('Não é você, é o segundo', fontsize=15, y=0.99)
rodape('Fontes: Bannister & Eerola 2018 (Frontiers in Psychology) · Bannister 2020 (Music & Science)')
plt.tight_layout(rect=(0, 0.03, 1, 0.95))
plt.savefig('dia-105-grafico-03-manipulacao.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")


### 💡 O Insight

**Entre 63% e 73% das pessoas sentem arrepio com música.**

Cinco pesquisas, 1.542 respondentes, quatro décadas entre a primeira e a última. Elas relatam entre 79% e 92%, e o modelo hierárquico chega a 85% como valor típico. Aplicado o fator de correção do projeto, sobram cerca de 7 em cada 10 pessoas.

Ou seja: é maioria. Não é sensibilidade excessiva, não é coisa sua.

E não é raro. Com 11,2% de chance por ocasião de escuta, o próximo arrepio chega, na mediana, na **sexta vez** que você põe música para tocar. Em 20 ocasiões, a chance de ao menos um é de **91%**. A conta bate com o observado: 81% tiveram ao menos um episódio ao longo de uma semana, o que corresponde a umas 14 ocasiões de escuta, cerca de duas por dia.

Mas o número que eu levo deste dia é o terceiro.

Se o arrepio fosse sobre quem escuta, ele estaria espalhado pela música. Não está. Ele se concentra em momentos que os pesquisadores conseguem apontar no relógio: a entrada de um instrumento, o crescendo, o clímax. E dá para mexer nisso.

Quando o trecho do arrepio é removido da gravação, os episódios caem 30%.

Quando o volume sobe 6 decibéis no ponto exato do clímax, o arrepio é multiplicado por **2,30**, com intervalo de 95% entre 1,52 e 3,49.

E a contraprova, que é a parte bonita: na peça sem clímax estrutural, o mesmo aumento de 6 decibéis não faz nada. Multiplicador de 0,94, intervalo entre 0,59 e 1,51.

Não é volume. É volume na hora certa.

*Qual foi a última música que te parou no meio do que você estava fazendo?*

---

### ⚠️ Limitações do Modelo

- **As camadas 1 e 2 são autorrelato.** Ninguém mediu a pele de 1.542 pessoas. É por isso que o ×0,80 entra, e entra em todas as proporções dessas duas camadas.
- **A camada 3 não leva ×0,80, e isso é uma escolha declarada.** São contagens de episódios em laboratório, com condutância da pele validando o botão. Aplicar o fator a medidas objetivas de experimento controlado seria justamente o erro que a regra do projeto proíbe.
- **O experimento de remoção é frágil.** Os autores relatam chi² = 3,85 e p = 0,049 com teste que respeita o desenho pareado. Refazendo a conta com um teste binomial simples sobre os mesmos 51 episódios, p = 0,26. A direção é a mesma nas três peças, mas com N=24 o resultado não é robusto à escolha do teste, e eu prefiro dizer isso a esconder.
- **A regressão de Poisson tem três pontos e dois parâmetros.** O ajuste é excelente por construção. O que sustenta o resultado não é o R², é o contraste com a peça de controle e o p de 0,0001 no coeficiente.
- **±6 dB é uma manipulação grande.** O estudo escolheu uma janela de 8 segundos no pico. Extrapolar o multiplicador para variações de volume da vida real não se sustenta.
- **Duas peças, dois estudos, mesmo laboratório.** Bannister assina os dois experimentos de manipulação. A direção é consistente com o catálogo de gatilhos que a revisão de 2021 monta a partir de 167 trabalhos, mas isso não substitui replicação independente.
- **As amostras de prevalência não são representativas.** Quem responde a uma pesquisa sobre reações à música provavelmente gosta mais de música do que a média. A própria revisão trata 90% como teto plausível.
- **A frequência vem de 106 universitários americanos em uma semana**, lida na revisão de 2021, que resume o estudo de 2014. O artigo original não foi aberto para este dia.
- **A simulação supõe ocasiões independentes e probabilidade igual para todo mundo.** Nenhuma das duas coisas é verdade: quem sente arrepio com frequência puxa a média para cima, e quem nunca sente puxa para baixo.
- **A genética não entra no cálculo**, só no contexto. E "herdável" não significa "determinado": as estimativas de 29% a 43% descrevem a variação entre pessoas numa população específica, não o destino de ninguém.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
